In [1]:
import psycopg2
import pandas as pd
import numpy as np
from datetime import datetime


In [2]:

DB_CONFIG = {
    "host": "172.23.106.87",
    "port": 5434,
    "database": "thingsboard",
    "user": "tbuser",
    "password": "tbpassword"
}

conn = psycopg2.connect(**DB_CONFIG)
print("Connected to DB")


Connected to DB


In [18]:
query = """
SELECT
    d.name AS device_name,
    to_timestamp(t.ts / 1000.0) AS time_utc,
    t.key,
    t.bool_v,
    t.long_v,
    t.dbl_v,
    t.str_v
FROM ts_kv t
JOIN device d ON d.id = t.entity_id
WHERE d.name = 'RackA_Device'
ORDER BY t.ts DESC
LIMIT 20;
"""


In [19]:
df = pd.read_sql(query, conn)
df


C:\Users\Admin\AppData\Local\Temp\ipykernel_16120\1258214632.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,device_name,time_utc,key,bool_v,long_v,dbl_v,str_v
0,RackA_Device,2025-12-15 12:19:31.403000+00:00,34,None,NaN,48.21,None
1,RackA_Device,2025-12-15 12:19:31.403000+00:00,68,None,NaN,34.96,None
2,RackA_Device,2025-12-15 12:19:31.403000+00:00,63,None,NaN,NaN,RackA
3,RackA_Device,2025-12-15 12:19:31.403000+00:00,64,None,NaN,77.17,None
4,RackA_Device,2025-12-15 12:19:31.403000+00:00,66,None,NaN,25.17,None
5,RackA_Device,2025-12-15 12:19:31.403000+00:00,65,None,NaN,1.27,None
6,RackA_Device,2025-12-15 12:19:31.403000+00:00,69,None,4315.0,NaN,None
7,RackA_Device,2025-12-15 12:19:31.403000+00:00,67,None,NaN,29.03,None
8,RackA_Device,2025-12-15 12:19:21.449000+00:00,63,None,NaN,NaN,RackA
9,RackA_Device,2025-12-15 12:19:21.449000+00:00,64,None,NaN,61.04,None


In [20]:
KEY_MAP = {
    63: "rack_id",
    64: "cpu_util",
    65: "power_kw",
    66: "ambient_temp_c",
    67: "inlet_temp_c",
    68: "exhaust_temp_c",
    69: "fan_speed_rpm",
    34: "humidity"
}


In [21]:
df["telemetry_key"] = df["key"].map(KEY_MAP)
df


,device_name,time_utc,key,bool_v,long_v,dbl_v,str_v,telemetry_key
0,RackA_Device,2025-12-15 12:19:31.403000+00:00,34,None,NaN,48.21,None,humidity
1,RackA_Device,2025-12-15 12:19:31.403000+00:00,68,None,NaN,34.96,None,exhaust_temp_c
2,RackA_Device,2025-12-15 12:19:31.403000+00:00,63,None,NaN,NaN,RackA,rack_id
3,RackA_Device,2025-12-15 12:19:31.403000+00:00,64,None,NaN,77.17,None,cpu_util
4,RackA_Device,2025-12-15 12:19:31.403000+00:00,66,None,NaN,25.17,None,ambient_temp_c
5,RackA_Device,2025-12-15 12:19:31.403000+00:00,65,None,NaN,1.27,None,power_kw
6,RackA_Device,2025-12-15 12:19:31.403000+00:00,69,None,4315.0,NaN,None,fan_speed_rpm
7,RackA_Device,2025-12-15 12:19:31.403000+00:00,67,None,NaN,29.03,None,inlet_temp_c
8,RackA_Device,2025-12-15 12:19:21.449000+00:00,63,None,NaN,NaN,RackA,rack_id
9,RackA_Device,2025-12-15 12:19:21.449000+00:00,64,None,NaN,61.04,None,cpu_util


In [22]:
unmapped_keys = df[df["telemetry_key"].isna()]["key"].unique()
print("Unmapped key IDs:", unmapped_keys)


Unmapped key IDs: []


In [23]:
df["value"] = (
    df["dbl_v"]
    .fillna(df["long_v"])
    .fillna(df["bool_v"])
)

df[["device_name", "time_utc", "telemetry_key", "value", "str_v"]]


C:\Users\Admin\AppData\Local\Temp\ipykernel_16120\1040475696.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df["bool_v"])


,device_name,time_utc,telemetry_key,value,str_v
0,RackA_Device,2025-12-15 12:19:31.403000+00:00,humidity,48.21,None
1,RackA_Device,2025-12-15 12:19:31.403000+00:00,exhaust_temp_c,34.96,None
2,RackA_Device,2025-12-15 12:19:31.403000+00:00,rack_id,NaN,RackA
3,RackA_Device,2025-12-15 12:19:31.403000+00:00,cpu_util,77.17,None
4,RackA_Device,2025-12-15 12:19:31.403000+00:00,ambient_temp_c,25.17,None
5,RackA_Device,2025-12-15 12:19:31.403000+00:00,power_kw,1.27,None
6,RackA_Device,2025-12-15 12:19:31.403000+00:00,fan_speed_rpm,4315.00,None
7,RackA_Device,2025-12-15 12:19:31.403000+00:00,inlet_temp_c,29.03,None
8,RackA_Device,2025-12-15 12:19:21.449000+00:00,rack_id,NaN,RackA
9,RackA_Device,2025-12-15 12:19:21.449000+00:00,cpu_util,61.04,None


In [24]:
# Separate rack_id (string) from numeric telemetry
rack_id_df = df[df["telemetry_key"] == "rack_id"][["time_utc", "str_v"]]
rack_id_df = rack_id_df.rename(columns={"str_v": "rack_id"})


In [25]:
numeric_df = df[df["telemetry_key"] != "rack_id"].copy()

numeric_df = numeric_df[[
    "device_name",
    "time_utc",
    "telemetry_key",
    "value"
]]

numeric_df


,device_name,time_utc,telemetry_key,value
0,RackA_Device,2025-12-15 12:19:31.403000+00:00,humidity,48.21
1,RackA_Device,2025-12-15 12:19:31.403000+00:00,exhaust_temp_c,34.96
3,RackA_Device,2025-12-15 12:19:31.403000+00:00,cpu_util,77.17
4,RackA_Device,2025-12-15 12:19:31.403000+00:00,ambient_temp_c,25.17
5,RackA_Device,2025-12-15 12:19:31.403000+00:00,power_kw,1.27
6,RackA_Device,2025-12-15 12:19:31.403000+00:00,fan_speed_rpm,4315.00
7,RackA_Device,2025-12-15 12:19:31.403000+00:00,inlet_temp_c,29.03
9,RackA_Device,2025-12-15 12:19:21.449000+00:00,cpu_util,61.04
10,RackA_Device,2025-12-15 12:19:21.449000+00:00,exhaust_temp_c,34.54
11,RackA_Device,2025-12-15 12:19:21.449000+00:00,inlet_temp_c,28.90


In [26]:
pivot_df = (
    numeric_df
    .pivot_table(
        index=["device_name", "time_utc"],
        columns="telemetry_key",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

pivot_df


telemetry_key,device_name,time_utc,ambient_temp_c,cpu_util,exhaust_temp_c,fan_speed_rpm,humidity,inlet_temp_c,power_kw
0,RackA_Device,2025-12-15 12:19:11.434000+00:00,NaN,75.67,NaN,4270.0,46.62,NaN,NaN
1,RackA_Device,2025-12-15 12:19:21.449000+00:00,25.85,61.04,34.54,3831.0,49.96,28.90,1.11
2,RackA_Device,2025-12-15 12:19:31.403000+00:00,25.17,77.17,34.96,4315.0,48.21,29.03,1.27
